# Entrenamiento del modelo de visión — Tiendita IA

**Tarea:** clasificar la foto de un producto (uno a la vez, como en el cobro). **Modelo base:** YOLO11n-cls (ImageNet). **Herramienta NVIDIA:** TensorRT.

Corre en el entorno GPU del **NVIDIA DLI** (Tesla T4). **Guarda este notebook con sus salidas**: es parte del registro de ejecución (entregable C).

⏱ **El tiempo de GPU es limitado.** Las fotos se extraen de los videos en la laptop (`preparar_videos.py`); aquí solo se entrena.

**Antes de empezar**, sube con el panel de archivos de Jupyter:
1. `vision.zip` — la carpeta `vision/` del repo (sin `raw/`, `videos/` ni `dataset/`).
2. `raw.zip` — las fotos: `raw/<sesion>/<clase>/*.jpg`.
3. `wheels_dli.zip` — paquetes de Python (el DLI no tiene internet).

## 0. Diagnóstico

In [ ]:
import sys
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!{sys.executable} -c "import urllib.request; urllib.request.urlopen('https://pypi.org', timeout=5); print('INTERNET OK')" || echo "SIN INTERNET: avisar para subir wheels a mano"
!df -h . | tail -1

## 1. Descomprimir código y fotos

In [ ]:
import zipfile, os
zipfile.ZipFile("vision.zip").extractall(".")
zipfile.ZipFile("raw.zip").extractall("vision")
print(sorted(os.listdir("vision/raw")), {s: sorted(os.listdir(f"vision/raw/{s}")) for s in os.listdir("vision/raw")})

## 2. Instalación sin internet (~1 min)
El DLI no tiene salida a internet, pero ya trae **torch 2.5.1+cu121**. El resto se instala desde `wheels_dli.zip` (paquetes descargados en la laptop para Python 3.10). Versiones fijadas para no romper el `sympy==1.13.1` que exige torch: onnxruntime 1.20.1 y onnxslim 0.1.98.

In [ ]:
import sys
import zipfile; zipfile.ZipFile("wheels_dli.zip").extractall(".")
!{sys.executable} -m pip install -q --no-index --find-links wheels_dli ultralytics onnx onnxslim onnxruntime pyyaml
# En el DLI no existe el comando `python`; las celdas usan {sys.executable} (el Python de este kernel).

In [ ]:
%cd vision
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))

## 3. División train / val / test por sesión
Ninguna sesión puede estar en dos conjuntos. Ajusta si sus sesiones se llaman distinto.

In [ ]:
import sys
!{sys.executable} split.py --train s1,s3 --test s2

## 4. Prueba de humo (2 épocas, ~1 min)
Verifica que todo corre antes del entrenamiento completo.

In [ ]:
import sys
!{sys.executable} train.py --nombre humo --epochs 2

## 5. Entrenamiento completo (~5–15 min en T4)

In [ ]:
import sys
!{sys.executable} train.py --nombre v1 --epochs 30

## 6. Evaluación con el conjunto de prueba

In [ ]:
import sys
!{sys.executable} eval.py --nombre v1

In [ ]:
from IPython.display import Image, display
display(Image("artefactos/v1/eval/val_test/confusion_matrix_normalized.png", width=650))

## 7. Optimización con TensorRT + benchmark

Si falla por `No module named 'tensorrt'` (el DLI no trae TensorRT y no hay internet), **sigan a la celda 8**: el `.onnx` y TensorRT se hacen en la laptop.

In [ ]:
import sys
!{sys.executable} export_trt.py --nombre v1

## 8. Empaquetar y DESCARGAR
Descarga `artefactos_v1.zip` (y este notebook guardado) **antes de cerrar el entorno**: los archivos pueden perderse.

In [ ]:
import shutil, json
shutil.rmtree("artefactos/humo", ignore_errors=True)
shutil.copy2("dataset/estadisticas.json", "artefactos/v1/estadisticas_datos.json")
print(shutil.make_archive("../artefactos_v1", "zip", ".", "artefactos/v1"))
print(open("artefactos/v1/registro.json").read()[:1500])